# 00 · 데이터셋 준비 및 가공

Phase 2 — `KorQuAD/squad_kor_v1` 원본을 로드하고 RAG 평가용 골든셋을 추출한다.

주요 로직은 `src/dataset.py`로 분리했고, 이 노트북은 **단위 확인용**으로 단계별 호출만 한다.

## 진행 단계
1. 원본 로드 — `dataset.load_korquad()`
2. 그룹핑/dedup → `data/docs/` — `build_docs` / `save_docs`
3. 골든셋 추출 → `data/golden_set.json` — `build_golden_set` / `save_golden_set`

In [ ]:
from src import config, dataset

## 1. 원본 로드

In [ ]:
ds = dataset.load_korquad()
train = ds["train"]
print(f"train: {len(ds['train'])} / validation: {len(ds['validation'])}")
train[0]

## 2. 그룹핑 & dedup

`title` 기준으로 묶고 중복 context를 제거(첫 등장 순서 보존 → 재현성)한 뒤 title별 .txt로 저장한다.

In [ ]:
docs = dataset.build_docs(train)
print(f"문서(title) 수: {len(docs)}")
print(f"고유 context 수: {sum(len(v) for v in docs.values())}")

In [ ]:
import statistics

sizes = [len(v) for v in docs.values()]
total_ctx = sum(sizes)
dup_ratio = 1 - total_ctx / len(train)
print(f"총 문서(title): {len(docs)}")
print(f"고유 context : {total_ctx}  (원본 {len(train)}행 중 중복 {dup_ratio:.1%} 제거)")
print(f"title당 context 수 — min {min(sizes)} / median {int(statistics.median(sizes))} / max {max(sizes)}")

In [ ]:
saved = dataset.save_docs(docs)
print(f"{len(saved)}개 문서를 {config.DOCS_DIR}/ 에 저장했습니다.")

# 예시 미리보기
sample = sorted(config.DOCS_DIR.glob('*.txt'))[0]
print('예시 파일:', sample.name)
print('-' * 60)
print(sample.read_text(encoding='utf-8')[:300], '...')

## 3. 골든셋 추출

`(question, ground_truth, reference_contexts)` 형식으로 `size=50`, `seed=42`(config 기본값) 고정 샘플링한다.
필터: context 100자 이상 + answer 비어있지 않음.

In [ ]:
import json

golden_set = dataset.build_golden_set(train)  # size=config.GOLDEN_SIZE, seed=config.SEED
print(f"샘플링: {len(golden_set)}개 (seed={config.SEED})")
print(json.dumps(golden_set[0], ensure_ascii=False, indent=2)[:400])

In [ ]:
path = dataset.save_golden_set(golden_set)
loaded = dataset.load_golden_set()
assert len(loaded) == config.GOLDEN_SIZE
assert all(d['question'] and d['ground_truth'] and d['reference_contexts'] for d in loaded)
print(f"{len(loaded)}개 골든셋을 {path} 에 저장했습니다.")
print(f"고유 질문: {len({d['question'] for d in loaded})} / {config.GOLDEN_SIZE}")